# 🟡 Atividade 02 — Prompt Engineering

**Conceito:** A qualidade da resposta depende diretamente da qualidade da pergunta.
Prompt Engineering é a arte de formular instruções claras e eficazes para a IA.

**Objetivo desta atividade:**
- Comparar prompts fracos vs. fortes na prática
- Testar as 4 técnicas principais: Persona, Chain-of-Thought, Few-Shot e Restrições
- Implementar um sistema de avaliação de prompts usando IA

---
> ⚠️ **Antes de começar:** substitua `SUA_CHAVE_AQUI` pela sua chave da API Anthropic.

## ⚙️ Setup

In [ ]:
%pip install anthropic -q

import anthropic
import os
import json
from typing import List, Dict, Any

# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURAÇÃO - Edite apenas esta seção
# ═══════════════════════════════════════════════════════════════════════════
os.environ["ANTHROPIC_API_KEY"] = "SUA_CHAVE_AQUI"  # ← Sua chave aqui
MODEL_NAME = "claude-sonnet-4-6"  # Modelo a usar
# ═══════════════════════════════════════════════════════════════════════════

client = anthropic.Anthropic()

def perguntar(prompt: str, system: str = "Você é um assistente útil.", 
              temperatura: float = 1.0, max_tokens: int = 500) -> str:
    """
    Função auxiliar para chamar a API do Claude.
    
    Args:
        prompt: A pergunta ou instrução para o modelo
        system: Mensagem de sistema que define o comportamento
        temperatura: Controla aleatoriedade (0.0 = determinístico, 1.0 = criativo)
        max_tokens: Limite máximo de tokens na resposta
    
    Returns:
        Texto da resposta do modelo
    """
    try:
        resposta = client.messages.create(
            model=MODEL_NAME,
            max_tokens=max_tokens,
            temperature=temperatura,
            system=system,
            messages=[{"role": "user", "content": prompt}]
        )
        return resposta.content[0].text
    except anthropic.AuthenticationError:
        return "❌ Erro: Chave de API inválida."
    except anthropic.RateLimitError:
        return "⚠️ Erro: Rate limit atingido. Aguarde."
    except Exception as e:
        return f"❌ Erro: {e}"

# Teste de conexão
resultado = perguntar("Diga apenas: pronto para começar!")
print("✅", resultado) if "pronto" in resultado.lower() else print(resultado)

---
## 📖 Parte 1 — Prompt fraco vs. Prompt forte

Vamos comparar respostas lado a lado para o mesmo tema.

In [ ]:
# ─── Par 1: Explicação de conceito ───
prompt_fraco_1  = "Me explica fotossíntese"

prompt_forte_1  = """Explique fotossíntese para um estudante do ensino médio.
Use uma analogia com algo do cotidiano para a parte química.
Estruture em:
1) O que é (1 frase)
2) Como funciona — com analogia
3) Por que importa para a vida na Terra
Máximo de 150 palavras."""

print("=" * 60)
print("PROMPT FRACO:")
print(perguntar(prompt_fraco_1))
print("\n" + "=" * 60)
print("PROMPT FORTE:")
print(perguntar(prompt_forte_1))

In [ ]:
# ─── Par 2: Geração criativa ───
prompt_fraco_2 = "Escreve um texto sobre tecnologia"

prompt_forte_2 = """Escreva um parágrafo de abertura para um artigo de opinião
voltado para jovens de 16 anos sobre como a IA pode ampliar desigualdades sociais.
Tom: reflexivo, mas acessível. Sem jargões técnicos.
Termine com uma pergunta que provoque o leitor a continuar lendo.
Máximo de 80 palavras."""

print("=" * 60)
print("PROMPT FRACO:")
print(perguntar(prompt_fraco_2))
print("\n" + "=" * 60)
print("PROMPT FORTE:")
print(perguntar(prompt_forte_2))

---
## 🔬 Parte 2 — As 5 Técnicas de Prompt Engineering

Vamos explorar as principais técnicas para criar prompts eficazes:
1. **Persona** — Definir um papel para o modelo assumir
2. **Chain-of-Thought** — Pedir raciocínio passo a passo
3. **Few-Shot** — Dar exemplos do formato desejado
4. **Restrições** — Delimitar formato, tom e conteúdo
5. **XML Tags** — Estruturar prompts complexos (muito eficaz no Claude!)

In [ ]:
# ─── Técnica 1: PERSONA ───
# Dá ao modelo um papel específico para assumir.
# Isso muda o vocabulário, o tom e o nível de detalhe da resposta.

system_padrao  = "Você é um assistente útil."
system_persona = """Você é a Prof. Ana, uma bióloga apaixonada que explica ciência
com analogias do cotidiano brasileiro. Você é animada, usa exemplos práticos
e sempre termina com uma curiosidade surpreendente."""

pergunta = "Como o DNA funciona?"

print("[SEM PERSONA]")
print(perguntar(pergunta, system=system_padrao))
print("\n[COM PERSONA - Prof. Ana]")
print(perguntar(pergunta, system=system_persona))

In [ ]:
# ─── Técnica 2: CHAIN-OF-THOUGHT (CoT) ───
# Pede ao modelo para raciocinar passo a passo antes de dar a resposta final.
# Melhora drasticamente problemas que precisam de lógica.

problema = "Tenho R$500 para gastar com material escolar. "\
           "Preciso de cadernos (R$15 cada), canetas (R$8 a caixa) "\
           "e uma calculadora científica (R$120). "\
           "Quero gastar pelo menos 60% em cadernos. Quanto de cada item devo comprar?"

# Sem CoT
print("[SEM CHAIN-OF-THOUGHT]")
print(perguntar(problema))

print("\n" + "-" * 60)

# Com CoT
problema_cot = problema + """

Resolva passo a passo:
1. Calcule o valor mínimo para cadernos (60% de R$500)
2. Verifique se a calculadora cabe no orçamento restante
3. Calcule quanto sobra para canetas
4. Dê a resposta final com a conta completa"""

print("[COM CHAIN-OF-THOUGHT]")
print(perguntar(problema_cot))

In [ ]:
# ─── Técnica 3: FEW-SHOT ───
# Dá exemplos do formato que você quer ANTES de pedir a resposta.
# O modelo aprende o padrão pelos exemplos.

prompt_zero_shot = "Classifique o sentimento: 'Finalmente acabou a prova!'"

prompt_few_shot = """Classifique o sentimento das frases. Siga este formato exato:

Frase: "Que dia lindo!"
Sentimento: alegria
Intensidade: alta
Emoji: 😄

Frase: "Perdi meu ônibus de novo."
Sentimento: frustração
Intensidade: média
Emoji: 😤

Frase: "Não sei se vou conseguir."
Sentimento: ansiedade
Intensidade: baixa
Emoji: 😟

Agora classifique:
Frase: "Finalmente acabou a prova!""""

print("[ZERO-SHOT (sem exemplos)]")
print(perguntar(prompt_zero_shot))
print("\n[FEW-SHOT (com exemplos)]")
print(perguntar(prompt_few_shot))

In [ ]:
# ─── Técnica 4: RESTRIÇÕES ───
# Delimita formato, tamanho, tom e o que NÃO pode aparecer.

# Sem restrições
print("[SEM RESTRIÇÕES]")
print(perguntar("Explique o que é inflação"))

print("\n" + "-" * 60)

# Com restrições
prompt_restrito = """Explique o que é inflação.
Restrições:
- Máximo de 3 frases
- Use linguagem de WhatsApp (informal, com emojis)
- Não use as palavras 'índice', 'econômico' ou 'monetário'
- Termine com um exemplo prático do supermercado"""

print("[COM RESTRIÇÕES]")
print(perguntar(prompt_restrito))

In [ ]:
# ─── Técnica 5: XML TAGS (Bonus - muito eficaz no Claude!) ───
# O Claude foi treinado para responder bem a estruturas com tags XML.
# Isso ajuda a organizar prompts complexos e delimitar seções claramente.

# Sem XML tags
prompt_sem_xml = """Analise este texto e me dê: um resumo de 2 frases, 
3 palavras-chave e o sentimento geral. O texto é: 
A inteligência artificial está revolucionando a medicina. 
Diagnósticos que antes levavam dias agora são feitos em minutos."""

# Com XML tags
prompt_com_xml = """Analise o texto abaixo e retorne as informações solicitadas.

<texto>
A inteligência artificial está revolucionando a medicina. 
Diagnósticos que antes levavam dias agora são feitos em minutos.
</texto>

<formato_resposta>
<resumo>2 frases resumindo o texto</resumo>
<palavras_chave>lista de 3 palavras-chave</palavras_chave>
<sentimento>positivo, negativo ou neutro</sentimento>
</formato_resposta>

Responda seguindo exatamente o formato acima."""

print("=" * 60)
print("[SEM XML TAGS]")
print(perguntar(prompt_sem_xml))
print("\n" + "=" * 60)
print("[COM XML TAGS]")
print(perguntar(prompt_com_xml))
print("\n💡 Dica: XML tags são especialmente úteis para separar instruções de dados!")

---
## 🏆 Desafio — Implemente o sistema `avaliar_prompt`

Técnica avançada: **LLM-as-judge** — usar uma IA para avaliar a resposta de outra IA.

Você vai construir uma função que:
1. Envia o prompt para o Claude e obtém uma resposta
2. Envia essa resposta para o Claude novamente, pedindo que ele a avalie
3. Retorna nota + justificativa em JSON

**Complete as partes marcadas com `# TODO`**

In [ ]:
def avaliar_prompt(prompt_usuario: str, criterios: List[str]) -> Dict[str, Any]:
    """
    Avalia a qualidade de um prompt usando o Claude como juiz.

    Args:
        prompt_usuario: o prompt a ser avaliado
        criterios: lista de critérios de avaliação

    Returns:
        dict com 'prompt', 'resposta', 'nota', 'justificativa', 'sugestao_melhoria'
    """

    # TODO 1: chame perguntar() para obter a resposta ao prompt_usuario
    resposta = None  # ← substitua

    # TODO 2: formate os critérios como uma string numerada
    # Exemplo: "1. clareza\n2. profundidade\n3. adequação ao público"
    # Dica: use enumerate() e join()
    criterios_formatados = None  # ← substitua

    # TODO 3: crie o prompt de avaliação usando as variáveis acima.
    # Use XML tags para estruturar! Exemplo:
    # <prompt_original>...</prompt_original>
    # <resposta_gerada>...</resposta_gerada>
    # <criterios>...</criterios>
    # Peça ao Claude para retornar SOMENTE um JSON válido com as chaves:
    # nota (1-10), justificativa (string), sugestao_melhoria (string)
    prompt_avaliacao = None  # ← substitua

    # TODO 4: chame perguntar() com o prompt_avaliacao
    # Use system="Você é um avaliador rigoroso. Responda SOMENTE em JSON válido."
    # e temperatura=0.0 para respostas mais consistentes
    avaliacao_raw = None  # ← substitua

    # TODO 5: faça o parse do JSON retornado
    # Dica: às vezes o modelo adiciona ```json ... ``` — remova isso antes do parse
    # Dica: use um bloco try/except para evitar erros de parsing
    try:
        # Limpa possíveis blocos de código markdown
        texto_limpo = avaliacao_raw or ""
        texto_limpo = texto_limpo.strip()
        if texto_limpo.startswith("```"):
            texto_limpo = texto_limpo.split("```")[1]
            if texto_limpo.startswith("json"):
                texto_limpo = texto_limpo[4:]
        avaliacao = json.loads(texto_limpo)
    except Exception:
        avaliacao = {"nota": 0, "justificativa": avaliacao_raw, "sugestao_melhoria": ""}

    return {
        "prompt":            prompt_usuario,
        "resposta":          resposta,
        "nota":              avaliacao.get("nota"),
        "justificativa":     avaliacao.get("justificativa"),
        "sugestao_melhoria": avaliacao.get("sugestao_melhoria"),
    }


# ─── Teste sua função ───
criterios = ["clareza", "profundidade", "adequação para estudante do ensino médio"]

prompts_para_avaliar = [
    "Me explica fotossíntese",
    prompt_forte_1,  # definido nas células anteriores
]

for p in prompts_para_avaliar:
    resultado = avaliar_prompt(p, criterios)
    print(f"PROMPT: {p[:60]}..." if len(p) > 60 else f"PROMPT: {p}")
    print(f"NOTA: {resultado['nota']}/10")
    print(f"JUSTIFICATIVA: {resultado['justificativa']}")
    print(f"SUGESTÃO: {resultado['sugestao_melhoria']}")
    print("─" * 70)

---
## 🎯 Desafio Extra — Combine todas as técnicas

Crie um prompt para uma matéria difícil da escola usando **Persona + Chain-of-Thought + Restrições** juntos.
Depois avalie seu próprio prompt com a função `avaliar_prompt`.

In [ ]:
# ─── Crie seu mega-prompt aqui ───

# Dica de estrutura:
# PERSONA: "Você é um professor de [matéria] que..."
# COT:     "Resolva passo a passo: 1)... 2)... 3)..."
# RESTRIÇÕES: "Máximo X palavras. Use... Não use..."

meu_mega_prompt = """
# TODO: escreva seu prompt aqui combinando as 3 técnicas
"""

# Avalie seu prompt
meus_criterios = ["clareza", "profundidade", "criatividade", "adequação ao público"]
resultado = avaliar_prompt(meu_mega_prompt, meus_criterios)

print("RESPOSTA GERADA:")
print(resultado["resposta"])
print(f"\nNOTA DO SEU PROMPT: {resultado['nota']}/10")
print(f"SUGESTÃO DE MELHORIA: {resultado['sugestao_melhoria']}")

---
## 📝 Respostas

Preencha o arquivo `respostas-02.md` com suas observações.

**Perguntas obrigatórias:**
1. Qual técnica gerou a maior diferença de qualidade entre o prompt fraco e o forte? Por quê?
2. O sistema LLM-as-judge avaliou os prompts de forma justa? Em quais casos ele errou?
3. Qual foi a nota do seu mega-prompt? O que você mudaria para melhorá-lo?

**Pergunta desafio:**

4. Se você pode usar IA para avaliar prompts, por que ainda precisamos de humanos para isso?